# 05-01 RunnablePassthrough: 데이터를 그대로 흘려보내기

## 학습 목표

이 노트북을 완료하면 다음을 할 수 있어요:

- `RunnablePassthrough()`로 입력을 변경하지 않고 그대로 다음 단계로 전달하는 방법을 설명할 수 있어요
- `RunnablePassthrough.assign()`으로 원본 데이터를 유지하면서 새로운 키를 동적으로 추가하는 방법을 구현할 수 있어요
- RAG(검색 증강 생성, Retrieval-Augmented Generation) 파이프라인에서 사용자 질문을 그대로 흘려보내는 패턴을 적용할 수 있어요

## 사전 지식

이 노트북을 시작하기 전에 다음 개념을 알고 있으면 좋아요:

- LCEL(LangChain Expression Language) 파이프 연산자(`|`) 사용법
- `RunnableParallel`의 기본 동작 방식
- Python 딕셔너리와 람다(lambda) 함수

---

`RunnablePassthrough`는 LCEL 파이프라인에서 입력 데이터를 처리하는 두 가지 방식을 제공해요:

- **그대로 전달**: 입력을 변경하지 않고 다음 단계로 전달
- **키 추가**: 원본 데이터를 유지하면서 새로운 키-값 쌍을 동적으로 추가

이 클래스는 `Runnable` 인터페이스를 구현하므로, 다른 `Runnable` 객체와 함께 파이프라인에서 사용할 수 있어요.

In [ ]:
# API 키를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv
import os

# API 키 정보 로드
load_dotenv()

# macOS에서 FAISS 사용 시 OpenMP 중복 로드 방지
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 모델 초기화
model = ChatOpenAI(model="gpt-4o-mini")


## 1. RunnablePassthrough() - 입력 그대로 전달

`RunnablePassthrough()`를 단독으로 호출하면 입력을 변경하지 않고 그대로 전달합니다.

일반적으로 `RunnableParallel`과 함께 사용되어 데이터를 맵의 새로운 키에 할당하는 데 활용됩니다.


In [ ]:
# ============================================================
# TODO: RunnableParallel과 RunnablePassthrough를 조합하여 체인을 완성하세요
# 힌트: passed=RunnablePassthrough(), extra=RunnablePassthrough.assign(...), modified=lambda ...
# 예상 결과: {'passed': {'num': 1}, 'extra': {'num': 1, 'mult': 3}, 'modified': 2}
# ============================================================

# ---------------------------------------------------
# RunnableParallel + RunnablePassthrough 세 가지 방식 비교
# ---------------------------------------------------

# 1단계: RunnableParallel 생성
# - passed: 원본 입력을 그대로 전달 (RunnablePassthrough())
# - extra: 원본에 "mult" 키를 추가 (RunnablePassthrough.assign())
# - modified: 입력값을 직접 변환하는 람다 함수


# 2단계: 실행 결과 확인


### 결과 분석

- `passed`: 원본 입력 `{'num': 1}`이 그대로 전달돼요
- `extra`: 원본 입력에 `mult` 키가 추가되어 `{'num': 1, 'mult': 3}` 형태로 반환돼요
- `modified`: `num`에 1을 더한 단순 변환 값 `2`가 반환돼요

**핵심 정리:**

| 방식 | 역할 | 출력 예시 |
|------|------|-----------|
| `RunnablePassthrough()` | 입력 그대로 통과 | `{'num': 1}` |
| `RunnablePassthrough.assign()` | 원본 유지 + 키 추가 | `{'num': 1, 'mult': 3}` |
| 람다 함수 | 값만 변환 | `2` |

> **실무 팁:** `RunnablePassthrough.assign()`은 체인 중간에서 원본을 잃지 않고 부가 정보를 붙이고 싶을 때 사용해요. 예를 들어 타임스탬프나 캐시 키를 덧붙이는 경우가 전형적인 활용 사례예요.

In [ ]:
# RunnablePassthrough.assign() 단독 사용 예제
# 
# assign()만 사용하면 원본 입력에 새로운 키만 추가됨

r = RunnablePassthrough.assign(mult=lambda x: x["num"] * 3)
result = r.invoke({"num": 1})

print("=" * 60)
print("📥 RunnablePassthrough.assign() 결과")
print("=" * 60)
print(f"입력: {{'num': 1}}")
print(f"출력: {result}")
print()
print("💡 원본 입력('num')은 유지되고, 'mult' 키만 추가됨")


## 2. RunnablePassthrough.assign() — 키를 추가하며 흘려보내기

`RunnablePassthrough.assign()`을 사용하면 원본 입력을 유지하면서 동적으로 새로운 키-값 쌍을 추가할 수 있어요.

**언제 사용하면 좋을까요?**

- 체인 중간에서 원본 데이터를 보존하면서 **계산된 값이나 메타데이터를 추가**해야 할 때
- 입력 데이터를 기반으로 **동적으로 새로운 정보를 생성**해야 할 때
- 예: 타임스탬프 추가, 조건부 값 계산, 외부 API 호출 결과 추가

아래 다이어그램은 `assign()`이 데이터를 어떻게 처리하는지 보여줘요:

```mermaid
flowchart LR
    A["입력<br/>{product_name, price, is_on_sale}"]:::input
    B["assign()<br/>discount_rate 계산<br/>current_time 추가"]:::process
    C["확장된 입력<br/>{product_name, price, is_on_sale,<br/>discount_rate, current_time}"]:::process
    D["ChatPromptTemplate"]:::process
    E["LLM"]:::process
    F["상품 설명 출력"]:::output

    A --> B --> C --> D --> E --> F

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
```

In [ ]:
# ============================================================
# TODO: RunnablePassthrough.assign()으로 동적 필드를 추가하는 체인을 구성하세요
# 힌트: chain = (RunnablePassthrough.assign(key=lambda x: ...) | prompt | model | StrOutputParser())
# 예상 결과: 할인율(30%)과 현재 시간이 포함된 상품 설명 생성
# ============================================================

# ---------------------------------------------------
# RunnablePassthrough.assign()으로 원본 입력에 새로운 정보 추가하기
# ---------------------------------------------------

from datetime import datetime

# 1단계: 프롬프트 템플릿 정의
# - assign()이 추가할 discount_rate, current_time 변수를 템플릿에 포함
product_prompt = ChatPromptTemplate.from_template(
    """다음 상품 정보를 바탕으로 매력적인 상품 설명을 작성해주세요.

상품명: {product_name}
가격: {price}원
할인율: {discount_rate}%
현재 시간: {current_time}

상품 설명:"""
)

# 2단계: RunnablePassthrough.assign()으로 'discount_rate'와 'current_time' 키를 동적으로 추가
#        - 원본 입력(product_name, price)은 그대로 유지
#        - assign()이 새로운 키들을 계산하여 추가


# 3단계: 실행 - 원본 입력에 할인율과 현재시간이 자동으로 추가되어 전달됨
original_input = {
    "product_name": "무선 이어폰",
    "price": 50000,
    "is_on_sale": True
}


## 섹션 전환 — RAG 패턴으로 확장하기

`assign()`으로 데이터를 확장하는 방법을 익혔으니, 이번에는 `RunnablePassthrough()`가 가장 많이 쓰이는 패턴인 RAG를 살펴볼게요.

RAG에서는 사용자의 질문이 두 군데에서 동시에 필요해요:
1. 벡터 저장소(Vector Store)에서 관련 문서를 검색하는 데
2. 최종 프롬프트에서 LLM에게 전달하는 데

`RunnablePassthrough()`는 질문을 복사하지 않고도 두 경로에 동시에 흘려보내는 역할을 해요.

```mermaid
flowchart TD
    Q["사용자 질문"]:::input
    R["Retriever<br/>문서 검색"]:::process
    P["RunnablePassthrough()<br/>질문 그대로 전달"]:::process
    C["context: 검색된 문서"]:::process
    QQ["question: 원본 질문"]:::process
    PT["ChatPromptTemplate"]:::process
    LLM["LLM"]:::process
    A["답변"]:::output

    Q --> R --> C --> PT
    Q --> P --> QQ --> PT
    PT --> LLM --> A

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
```

In [ ]:
# ============================================================
# TODO: RAG 체인에서 RunnablePassthrough()로 질문을 전달하는 구조를 완성하세요
# 힌트: {"context": retriever | format_docs, "question": RunnablePassthrough()} | prompt | model | ...
# 예상 결과: "김민수의 직업은 머신러닝 엔지니어입니다." 형태의 답변
# ============================================================

from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

# 1단계: FAISS 벡터 저장소 생성 (예제 데이터: 회사 직원 정보)
vectorstore = FAISS.from_texts(
    [
        "김민수는 AI 스타트업에서 머신러닝 엔지니어로 근무하고 있습니다.",
        "이지은은 같은 회사에서 프론트엔드 개발자로 일하고 있습니다.",
        "김민수의 주요 업무는 자연어 처리 모델 개발입니다.",
        "이지은의 주요 업무는 React 기반 웹 애플리케이션 개발입니다.",
        "두 사람은 협업하여 AI 기반 웹 서비스를 개발하고 있습니다.",
    ],
    embedding=OpenAIEmbeddings(),
)

# 2단계: 벡터 저장소를 검색기로 변환
# as_retriever(): VectorStore → Retriever (Runnable 인터페이스 구현)
retriever = vectorstore.as_retriever()

# RAG 프롬프트 템플릿 정의
template = """다음 컨텍스트를 바탕으로 질문에 답변해주세요.
컨텍스트에 없는 내용은 추측하지 마세요.

컨텍스트:
{context}

질문: {question}

답변:"""

prompt = ChatPromptTemplate.from_template(template)

# 3단계: 검색된 문서를 포맷팅하는 함수
def format_docs(docs):
    """검색된 문서들을 하나의 문자열로 결합"""
    return "\n".join([doc.page_content for doc in docs])

# 4단계: RAG 체인 구성
# - context: retriever로 검색한 문서들을 포맷팅
# - question: RunnablePassthrough()로 사용자 질문을 변경 없이 그대로 전달


In [ ]:
# RAG 체인 실행 예제 1
question1 = "김민수의 직업은 무엇입니까?"
answer1 = retrieval_chain.invoke(question1)

print("=" * 60)
print("📥 질문 1")
print("=" * 60)
print(f"질문: {question1}")
print()
print("📤 답변:")
print(answer1)


In [ ]:
# RAG 체인 실행 예제 2
question2 = "이지은의 주요 업무는 무엇입니까?"
answer2 = retrieval_chain.invoke(question2)

print("=" * 60)
print("📥 질문 2")
print("=" * 60)
print(f"질문: {question2}")
print()
print("📤 답변:")
print(answer2)


## 마무리 요약

### RunnablePassthrough의 두 가지 사용법

| 사용법 | 역할 | 주요 사용 사례 |
|--------|------|----------------|
| `RunnablePassthrough()` | 입력을 변경 없이 그대로 전달 | RAG에서 질문 전달, 데이터 흐름 유지 |
| `RunnablePassthrough.assign()` | 원본 데이터 유지 + 새로운 키 추가 | 타임스탬프, 조건부 값, 메타데이터 추가 |

### 핵심 요점

- `RunnablePassthrough()`는 입력을 변경하지 않고 그대로 전달해요
- `RunnablePassthrough.assign()`은 원본 데이터 손실 없이 추가 정보를 전달해요
- RAG 시스템에서 사용자 질문을 프롬프트에 그대로 흘려보낼 때 필수적으로 사용해요
- `RunnableParallel`과 함께 사용하면 여러 경로로 데이터를 분기할 수 있어요

### 다음 노트북 예고

다음 노트북(`02-Inspect-Runnables.ipynb`)에서는 지금까지 구성한 체인의 내부 구조를 시각적으로 확인하는 방법을 배워요. `get_graph()`, `print_ascii()` 등 디버깅에 유용한 도구를 살펴볼게요.